**Cell #01**

# RAG11 — Stage 2 (Nutriciology): Five Funny Questions, Serious Answers

People do not ask textbook questions. They ask *"can I survive on pizza if the sauce counts as a vegetable?"*, in
slang, with emoji, sometimes in French. This notebook sends five such questions through the full nutrition
pipeline and shows that the answers stay grounded in the textbooks anyway.

**Speaking language.** `speaking_language = "EN"` (cell #02): questions may be in *any* language, register or
script, and every answer is written in English. What makes that smooth:

1. **Question understanding** (`prepare_question()`, one small Claude call before retrieval) detects the language,
   translates the question into English, and rewrites the joke into a neutral, keyword-rich **search query**
   ("Can a diet of only pizza meet nutrient needs? vegetable servings, tomato sauce"). Retrieval searches with that
   query, because jokes, slang and emoji are noise to both the embedding and the keyword search. The answering
   model still sees the original question and its English translation, so it can answer in the same spirit.
2. **An enforced answer language** (`ask_question(answer_language=...)`): an explicit instruction in the system
   prompt, so the answer is English even for the French question.
3. **The full retrieval stack**: hybrid search + multi-query splitting + reranking + parent-chunk expansion.

The Question / Answer cards come from `show_qa()` in `reusable_code/display.py`, the same design as
`stage2_ask_examples7_ys.ipynb`.

**Before running this notebook**: `stage1_2` must have loaded the nutrition textbooks (`stage1_9` shows PASS).

In [1]:
# Cell #02
from reusable_code import (
    init_clients, ask_question, prepare_question, show_qa, show_summary, language_name,
    SYSTEM_PROMPT, GENERATION_MODEL,
)
from reusable_code.env import optional_env

# Every answer is written in this language, whatever language/script the question is in
# (ISO 639-1 code: EN, FR, DE, ES, HI, ...). The default comes from SPEAKING_LANGUAGE in .env.
speaking_language = "EN"

clients = init_clients()
print("Clients ready. Supabase project:", optional_env("PUBLIC_SUPABASE_URL"), "| model:", GENERATION_MODEL)
print("Speaking language:", speaking_language, f"({language_name(speaking_language)})")

Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co | model: claude-sonnet-5
Speaking language: EN (English)


**Cell #03**

## The pipeline

The nutrition system prompt is reused, with one addition for playful questions: answer the real question underneath,
seriously and only from the excerpts. A light touch of humour in an opening sentence is fine; inventing facts to be
funny is not.

In [2]:
# Cell #04
NUTRI_SYSTEM_PROMPT = SYSTEM_PROMPT + """

The question may be playful, use slang or emoji, or be written in another language. Answer the real nutrition \
question underneath, seriously and accurately, using only the excerpts. One light, good-natured opening sentence \
is welcome; never sacrifice accuracy for a joke and never add a fact that is not in the excerpts."""

NUTRI_CORPUS_HINT = "English-language clinical and sports nutrition textbooks"


def ask_nutri(prepared, **overrides) -> dict:
    """Run one prepared question (a PreparedQuestion) through the full nutrition pipeline."""
    options = dict(
        match_count=5,
        use_hybrid=True,            # exact terms (kcal, vitamin names) via keyword search + meaning via embeddings
        use_multi_query=True,       # jokes often hide two questions in one
        use_hyde=False,
        use_rerank=True,
        expand_to_parents=True,     # show the whole textbook section around each matched chunk
        system_prompt=NUTRI_SYSTEM_PROMPT,
        answer_language=speaking_language,
        retrieval_query=prepared.retrieval_query,
    )
    options.update(overrides)
    return ask_question(prepared.llm_question, **options)

**Cell #05**

## The five funny questions

Each is a joke wrapped around a real textbook topic, and each stresses the pipeline differently:

1. **The pizza defence** (English): nutrient adequacy and what counts as a vegetable serving.
2. **The espresso roommate** (English, two questions in one): does coffee count toward hydration, and does caffeine
   dehydrate? A natural case for multi-query splitting.
3. **Le chocolat noir** (French): plant foods versus "vegetables", asked in French; the answer must come back in English.
4. **Gummy vitamins** (slang, emoji, typos): vitamin toxicity and tolerable upper intake levels, hidden under
   "bro" and 💪.
5. **The midnight fridge** (a genuine yes/no): do calories eaten unobserved count? Energy balance, with a clean
   `Short answer: No`.

In [3]:
# Cell #06
FUNNY_QUESTIONS = [
    # 1. Nutrient adequacy + what counts as a vegetable.
    "Can I survive on pizza alone if I keep insisting that tomato sauce counts as a vegetable?",
    # 2. Two questions in one: hydration credit for coffee + caffeine and dehydration.
    "My roommate swears six espressos count as my eight glasses of water. Is he right, or will the caffeine "
    "dehydrate me until I turn into a raisin?",
    # 3. French: plant foods vs vegetables. The answer must come back in English.
    "Le chocolat noir compte-t-il comme une portion de légumes, vu que le cacao est une plante ? "
    "Mon cerveau veut savoir avant le dessert.",
    # 4. Slang + emoji + a dangerous idea: vitamin toxicity / tolerable upper intake.
    "bro are gummy vitamins 🍬 candy or medicine?? can i just eat like 20 of them to be extra healthy 💪😂",
    # 5. A genuine yes/no: energy balance does not care who is watching.
    "Do calories eaten standing in front of the open fridge at midnight not count, because nobody was watching?",
]

**Cell #07**

## Step 1 — what the funny questions turn into

`prepare_question()` runs once per question (one small Claude call each). The table shows the detected language, the
English translation and the neutral **search query** that retrieval uses instead of the raw joke. If that call ever
fails, the original question is used unchanged, so a question is never lost.

In [4]:
# Cell #08
funny_prepared = [prepare_question(q, language=speaking_language, corpus_hint=NUTRI_CORPUS_HINT) for q in FUNNY_QUESTIONS]

show_summary(
    [(i, p.language or "?", p.translation or "(same)", p.retrieval_query) for i, p in enumerate(funny_prepared, start=1)],
    ["#", "language", "understood as", "search query used for retrieval"],
)

#,language,understood as,search query used for retrieval
1,EN,(same),"Is a diet consisting only of pizza nutritionally adequate, and does tomato sauce count as a vegetable serving?"
2,EN,(same),Does caffeine intake from coffee cause dehydration or contribute to daily fluid and water balance?
3,FR,"Does dark chocolate count as a vegetable serving, since cacao is a plant? My brain wants to know before dessert.",Does dark chocolate or cacao count as a vegetable serving in nutrition guidelines?
4,EN,Are gummy vitamins candy or medicine? Can I just eat like 20 of them to be extra healthy?,"Are gummy vitamins classified as dietary supplements or candy, and what are the risks of consuming excessive amounts of vitamin supplements?"
5,EN,(same),Does eating context or awareness affect calorie intake and weight gain? Nighttime snacking and caloric intake


**Cell #09**

## Step 2 — run all 5 questions

Each prepared question goes through `ask_nutri()` and is displayed as a **Question / Answer** card: the answer's
Markdown is rendered as paragraphs and lists, a genuine yes/no shows as a badge, and the card has `height: auto`, so
the full answer is always visible. The footer shows the excerpts kept out of the candidates the reranker saw, the
textbook pages, and the sub-questions multi-query produced.

(If your Jupyter front end puts a long output in a scrolling box, right-click it and choose "Disable Scrolling for
Outputs"; JupyterLab and PyCharm never do.)

In [5]:
# Cell #10
funny_results = []
for number, (question, prepared) in enumerate(zip(FUNNY_QUESTIONS, funny_prepared), start=1):
    result = ask_nutri(prepared)
    funny_results.append(result)
    show_qa(number, question, result, prepared)

**Cell #11**

## Summary table

In [6]:
# Cell #12
show_summary(
    [
        (i, p.language or "?", r["short_answer"] or "n/a", len(r["subquestions"] or []), r["chunks_used"],
         ", ".join(r["source_keys"]), q[:70] + ("..." if len(q) > 70 else ""))
        for i, (q, p, r) in enumerate(zip(FUNNY_QUESTIONS, funny_prepared, funny_results), start=1)
    ],
    ["#", "question language", "short answer", "sub-Qs", "excerpts", "sources", "question"],
)

#,question language,short answer,sub-Qs,excerpts,sources,question
1,EN,No,2,4,"source1, source11, source5, source6",Can I survive on pizza alone if I keep insisting that tomato sauce cou...
2,EN,No,2,3,source11,My roommate swears six espressos count as my eight glasses of water. I...
3,FR,No,2,4,"source11, source17, source3","Le chocolat noir compte-t-il comme une portion de légumes, vu que le c..."
4,EN,No,2,4,"source11, source13, source17, source2",bro are gummy vitamins 🍬 candy or medicine?? can i just eat like 20 of...
5,EN,No,2,5,"source11, source12, source5, source6",Do calories eaten standing in front of the open fridge at midnight not...


**Cell #13**

## What does question understanding buy? (retrieval only)

Compares **hybrid retrieval alone** for the emoji question (#4), once with the raw text and once with the search query
from step 1, showing which half of the hybrid search (dense or keyword) found each chunk and where it came from. The
keyword half in particular is easily thrown off by "bro", "gummy" and emoji, while the search query names the real
topic (vitamin toxicity, tolerable upper intake level).

In [7]:
# Cell #14
from reusable_code import hybrid_search

for label, text in [("raw question", FUNNY_QUESTIONS[3]), ("search query", funny_prepared[3].retrieval_query)]:
    rows = hybrid_search(text, match_count=5)
    print(f"--- {label}: {text[:100]}")
    for r in rows:
        first_line = r["rowJSON"]["text"].split("\n", 1)[0][:100]
        print(f"  dense#{r['dense_rank']!s:<4} keyword#{r['keyword_rank']!s:<5} {first_line}")

--- raw question: bro are gummy vitamins 🍬 candy or medicine?? can i just eat like 20 of them to be extra healthy 💪😂
  dense#8    keyword#1     [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Supple
  dense#18   keyword#6     [Source: _OceanofPDF.com_Encyclopedia_of_foods_-_Mayo_Clinic.pdf | Section: Chapter 2. The Nutrients
  dense#1    keyword#None  [Source: Nutrition_for_Nurses-WEB_260913_200839.pdf | Section: 3.3 Supplements | Pages 67-70]
  dense#2    keyword#None  [Source: _OceanofPDF.com_Encyclopedia_of_foods_-_Mayo_Clinic.pdf | Section: Chapter 2. The Nutrients
  dense#None keyword#2     [Source: Nutrition-Science-and-Everyday-Application-1773787282.pdf | Section: Tools for Achieving a 
--- search query: Are gummy vitamins classified as dietary supplements or candy, and what are the risks of consuming e
  dense#1    keyword#6     [Source: _OceanofPDF.com_Encyclopedia_of_foods_-_Mayo_Clinic.pdf | Section: Chapter 2. The Nutrients
  d

**Cell #15**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock recovery and conflict resolution).

In [8]:
# Cell #16
from reusable_code import save_to_github

save_to_github("stage2_ask_examples8_nutriciology.ipynb - five funny questions, speaking_language EN")

  RAG11 -> GitHub Robust Sync Utility
  Directory : /Users/mgtimber/CV26/RAG11
  Remote URL: https://github.com/fotomain/RAG11-nutriciology.git
[0/5] Checking repository health & clearing stale locks...
[1/5] Git repository verified.
[2/5] Origin remote verified: https://github.com/fotomain/RAG11-nutriciology.git
[3/5] Staging workspace files...
[4/5] Committing changes: "stage2_ask_examples8_nutriciology.ipynb - five funny questions, speaking_language EN"
[main d36e720] stage2_ask_examples8_nutriciology.ipynb - five funny questions, speaking_language EN
 10 files changed, 871 insertions(+), 461 deletions(-)
 create mode 100644 reusable_code/display.py
 create mode 100644 reusable_code/language.py
 create mode 100644 stage2_ask_examples8_nutriciology.ipynb
[5/5] Synchronizing with GitHub (main)...
      Pushing to origin main (attempt 1/3)...
branch 'main' set up to track 'origin/main'.

  Successfully synchronized with GitHub!
  Branch    : main
  Commit    : d36e720
  Repository: htt

To https://github.com/fotomain/RAG11-nutriciology.git
   4a2825e..d36e720  main -> main



True